# Table Transformer (DETR-R18, PubTables-1M) — DIMER table detection tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/table-transformer-detection-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/table-transformer-detection-pipeline/blob/main/tutorials/table_transformer_detection_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-microsoft%2Ftable--transformer--detection-ffcc4d?style=flat)](https://huggingface.co/microsoft/table-transformer-detection) [![Upstream](https://img.shields.io/badge/Upstream-microsoft%2Ftable--transformer-181717?style=flat&logo=github&logoColor=white)](https://github.com/microsoft/table-transformer) [![arXiv](https://img.shields.io/badge/arXiv-2110.00061-b31b1b.svg)](https://arxiv.org/abs/2110.00061)

**Profile:** `TASK-INFERENCE`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** table detection on document page images (boxes labelled `table` or `table rotated`) using the pinned `microsoft/table-transformer-detection` weights

**This notebook is standalone.** It carries the repository's pipeline module (`src/table_transformer_detection_pipeline/pipeline.py` at revision `b4e37a9a1cdf`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `2357cbe2b5a5d1c03e54f32764f06058933b65ab` (~115 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned snapshot, obtains the tutorial sample automatically, validates it into an input manifest before the model runs, runs the task locally in this kernel, writes the evaluation report, and exports machine-readable outputs with provenance. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5).

**Bring Your Own Data:** After the sample workflow completes, set `USE_BYOD = True` in the sample cell and re-run from that cell to supply your own input. It passes through the same notebook-local validation, task, evaluation-report and export cells as the sample; the expected input format, the ceilings and the privacy guidance are stated in the Prerequisites and in the sample cell, and the upload stays inside this runtime. BYOD is optional and never part of the default path.

At inference the DETR-style model reads one page image resized to 800 px on its shortest edge, runs a ResNet-18 backbone and a 6-layer encoder–decoder, and emits exactly 15 query proposals, each a box and a softmax over `table`, `table rotated` and *no object*; the processor keeps the queries whose class score reaches the threshold and maps their boxes back to input pixels. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting happens in this notebook — the upstream checkpoint supplies the weights and image-processor configuration, and the carried module adds snapshot verification, the input contract, a fixed output contract and the `box_iou`, `validate_inputs` and `evaluation_report` helpers. The default sample is a page rendered in code with two ruled tables whose drawn boxes serve as references; its `box_iou` values are demonstration (plumbing) evidence for one page, not a detection benchmark.

**Learning objectives:** install the pinned runtime, read what the carried pipeline module guarantees, resolve and digest-verify the immutable upstream model revision, render a synthetic page with reference table boxes (or upload your own page) and validate it into an input manifest, run the supported task, read the class scores and the caller-owned threshold correctly, exercise an optional BYOD path, produce an evaluation report that is `sample-sanity` with `box_iou` only when reference boxes exist and `not-measurable` otherwise, and export machine-readable detections plus an annotated image and provenance.

**This notebook does not demonstrate:** table *structure* recognition (rows, columns, cells — the sibling `table-transformer-structure-pipeline` covers it), OCR or cell text extraction, figure/chart/text-block detection, page rotation correction, mAP or precision/recall evaluation (which needs a labelled page set), or any training. The model was trained on PubTables-1M PDF renders; scans, photographs of pages and non-Latin layouts are outside what this notebook measures.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU and uses CUDA automatically when available; inference is float32 on both. CPU is adequate: the repository's model card records 4.8 s to load and 0.17 s per `detect` on the 850×1100 synthetic page in the Windows venv (Intel Core Ultra 9 275HX). The pinned `torch==2.14.0` install and the 115 MB checkpoint are the large downloads of the run.
- **Knowledge:** basic Python and PIL; what a bounding box in xyxy pixel coordinates is; what intersection-over-union measures.
- **Data:** the default sample is a deterministic 850×1100 page rendered in code with Pillow's bundled font — a heading, two paragraphs, a wide 8×5 table and a small 5×3 table — so nothing is downloaded and no private data is needed. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Expected BYOD input: one page image decodable by Pillow (PNG/JPEG/WebP and similar), any colour mode, sides between 16 and 4096 px. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `microsoft/table-transformer-detection` snapshot (~115 MB in total) at revision `2357cbe2b5a5…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers`, `timm` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'transformers==4.57.6',
    'timm==1.0.29',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'huggingface-hub==0.36.2',
]
NOTEBOOK_SOURCE = {
    'repository': 'table-transformer-detection-pipeline',
    'repository_revision': 'b4e37a9a1cdf4fc6595f88a8f2f661a059995db7',
    'embedded_module': 'src/table_transformer_detection_pipeline/pipeline.py',
    'embedded_modules': ['src/table_transformer_detection_pipeline/pipeline.py'],
    'module_sha256': '16cbd0753fa7bfb0664df367ac3f044cb1414ef1b99e5a53d0e5952d6219a734',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers, timm
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'timm': timm.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/table_transformer_detection_pipeline/` @ `b4e37a9a1cdf`)

The next 1 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/1:** `src/table_transformer_detection_pipeline/pipeline.py`

In [ ]:
from __future__ import annotations

import hashlib
import json
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

from PIL import Image

MODEL_ID = "microsoft/table-transformer-detection"
MODEL_REVISION = "2357cbe2b5a5d1c03e54f32764f06058933b65ab"
MODEL_LICENSE = "mit"
MODEL_KEY = "table-transformer-detection"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# The two classes the checkpoint was fine-tuned on (config.json id2label). A "table rotated" box is a
# table whose text runs vertically; the pipeline returns the label as-is and does not rotate anything.
LABELS = ("table", "table rotated")
# Detection threshold: the value the Transformers Table Transformer documentation example passes to
# post_process_object_detection (threshold=0.9). It gates a softmax class score over 15 DETR queries
# that was not calibrated for any document domain; the deployment owns tuning it on labelled pages.
DETECTION_THRESHOLD = 0.9
# The checkpoint's DETR decoder emits exactly num_queries proposals per page (config.json), so no
# page can yield more than this many boxes.
MAX_DETECTIONS = 15
# Input ceilings. The processor resizes the shortest edge to 800 px with the longest capped at 800
# (preprocessor_config.json `size`/`max_size`), so image cost is bounded whatever the caller sends;
# the side ceiling only guards memory during decoding and resizing.
MAX_IMAGE_SIDE = 4096
MIN_IMAGE_SIDE = 16


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def box_iou(a: Sequence[float], b: Sequence[float]) -> float:
    """Intersection-over-union of two xyxy pixel boxes; the building block for any caller-side mAP."""
    if len(a) != 4 or len(b) != 4:
        raise ValueError("boxes must be [x0, y0, x1, y1]")
    if a[2] < a[0] or a[3] < a[1] or b[2] < b[0] or b[3] < b[1]:
        raise ValueError("boxes must satisfy x0 <= x1 and y0 <= y1")
    inter_w = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    inter_h = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = inter_w * inter_h
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return float(inter / union) if union > 0 else 0.0


def validate_image(image: Any) -> Image.Image:
    if not isinstance(image, Image.Image):
        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")
    width, height = image.size
    if min(width, height) < MIN_IMAGE_SIDE:
        raise ValueError(f"image side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"image side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    return image.convert("RGB")


def _check_threshold(value: Any) -> float:
    if isinstance(value, bool) or not isinstance(value, int | float) or not 0.0 <= value <= 1.0:
        raise ValueError(f"threshold must be a number in [0, 1], got {value!r}")
    return float(value)


INPUT_SCHEMA: dict[str, Any] = {
    "input": "one page image as PIL.Image.Image (any mode, converted to RGB): a rendered page or a scan",
    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "threshold": [0.0, 1.0],
    "labels": list(LABELS),
    "max_detections": MAX_DETECTIONS,
    "preprocessing": (
        "image converted to RGB; the processor resizes to shortest edge 800 px (longest edge capped at "
        "800), normalises with ImageNet mean/std, and returned boxes are mapped back to input pixels"
    ),
}


def _check_inputs(image: Any, threshold: Any) -> tuple[Image.Image, float]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the checked request.

    ``detect`` and ``validate_inputs`` both route through this function so their acceptance
    criteria cannot diverge.
    """
    return validate_image(image), _check_threshold(threshold)


def validate_inputs(
    image: Image.Image,
    *,
    threshold: float = DETECTION_THRESHOLD,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict).

    Rejection is reported by raising exactly as ``detect`` would; a caller that wants the finding
    recorded catches the exception and stores ``str(exc)`` under ``findings``.
    """
    _rgb, checked = _check_inputs(image, threshold)
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry (detect takes one page image)")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [{"id": names[0] if names else "image-0", "mode": image.mode, "size": list(image.size)}],
        "threshold": checked,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any],
    ground_truth_boxes: Sequence[Sequence[float]] | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``ground_truth_boxes`` (xyxy reference boxes of the tables on the page) the report carries
    one ``box_iou`` entry per reference — the best-overlapping detection — as sample-sanity geometry
    evidence; without them the verdict is ``not-measurable`` and the report says what labelled data
    would make the task measurable.
    """
    detections = list(result["detections"])
    base = {
        "task": "table detection on document page images",
        "decision_rule": (
            "a DETR query survives when its softmax score for `table` or `table rotated` reaches the "
            "threshold; the score is a class probability under the model's own softmax, not a calibrated "
            "estimate for the deployment's pages"
        ),
        "threshold": result.get("threshold", DETECTION_THRESHOLD),
        "sample_kind": sample_kind,
        "n_detections": len(detections),
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if not ground_truth_boxes:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no ground-truth table boxes were supplied for the evaluated page",
            "needs": (
                "labelled table boxes on your own pages, scored per table with box_iou and aggregated into "
                "precision/recall or mean average precision at a stated IoU threshold; no such labelled set "
                "ships with this repository"
            ),
        }
    metrics = []
    for index, box in enumerate(ground_truth_boxes):
        ious = [box_iou(det["box"], box) for det in detections]
        best = max(range(len(ious)), key=ious.__getitem__) if ious else None
        metrics.append(
            {
                "id": "box_iou",
                "reference": f"table-{index}",
                "value": ious[best] if best is not None else 0.0,
                "matched_label": detections[best]["label"] if best is not None else None,
                "estimation": "one reference box per table on a single page, no dispersion estimate",
            }
        )
    return {
        **base,
        "metrics": metrics,
        "verdict": "sample-sanity",
        "reason": (
            f"{len(metrics)} reference box(es) on one tutorial page; geometry sanity evidence, "
            "not a detection benchmark"
        ),
        "needs": (
            "a labelled page set from the deployment domain (scans, renders, layouts) for any "
            "mean-average-precision or precision/recall claim"
        ),
    }


@dataclass
class TableTransformerDetectionPipeline:
    """Table detection on document page images over the pinned Table Transformer (DETR-R18) checkpoint."""

    _runner: Callable[[Image.Image, float], list[dict[str, Any]]]
    device: str

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> TableTransformerDetectionPipeline:
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            source, kwargs = str(root), {"local_files_only": True}
        elif allow_download:
            source, kwargs = MODEL_ID, {}
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage {MODEL_ID}@{MODEL_REVISION} under weights/{MODEL_KEY}"
            )
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import AutoImageProcessor, TableTransformerForObjectDetection

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        processor = AutoImageProcessor.from_pretrained(
            source, revision=MODEL_REVISION, trust_remote_code=False, **kwargs
        )
        # config.json says use_pretrained_backbone=true, which would make the timm ResNet-18 backbone
        # fetch ImageNet weights from the Hub at construction time — an unpinned download that the
        # checkpoint immediately overwrites. The backbone weights are in model.safetensors; refuse it.
        model = TableTransformerForObjectDetection.from_pretrained(
            source,
            revision=MODEL_REVISION,
            trust_remote_code=False,
            use_pretrained_backbone=False,
            **kwargs,
        )
        model = model.to(resolved_device).eval()
        id2label = {int(k): v for k, v in model.config.id2label.items()}

        def runner(image: Image.Image, threshold: float) -> list[dict]:
            inputs = processor(images=image, return_tensors="pt").to(resolved_device)
            with torch.inference_mode():
                outputs = model(**inputs)
            result = processor.post_process_object_detection(
                outputs, threshold=threshold, target_sizes=[image.size[::-1]]
            )[0]
            return [
                {
                    "box": [float(v) for v in box.tolist()],
                    "label": id2label[int(label)],
                    "score": float(score),
                }
                for box, label, score in zip(result["boxes"], result["labels"], result["scores"], strict=True)
            ]

        return cls(runner, resolved_device)

    def detect(self, image: Image.Image, *, threshold: float = DETECTION_THRESHOLD) -> dict[str, Any]:
        """Detect tables on one page image; boxes are xyxy pixel coordinates in the input image."""
        rgb, checked = _check_inputs(image, threshold)
        detections = self._runner(rgb, checked)
        if len(detections) > MAX_DETECTIONS:
            raise RuntimeError(
                f"backend returned {len(detections)} detections > num_queries {MAX_DETECTIONS}"
            )
        for det in detections:
            if set(det) != {"box", "label", "score"} or len(det["box"]) != 4 or det["label"] not in LABELS:
                raise RuntimeError(f"backend returned a malformed detection: {det!r}")
        return {
            "detections": sorted(detections, key=lambda d: -d["score"]),
            "threshold": checked,
            "width": rgb.width,
            "height": rgb.height,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `4`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `2357cbe2b5a5…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `TableTransformerDetectionPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "table-transformer-detection",
  "modelId": "microsoft/table-transformer-detection",
  "revision": "2357cbe2b5a5d1c03e54f32764f06058933b65ab",
  "files": [
    {
      "path": "README.md",
      "bytes": 1174,
      "sha256": "c91e7f8199313c4d24b09e73a2f6df3141268666969346cb7b94ccb59900f5dd"
    },
    {
      "path": "config.json",
      "bytes": 1228,
      "sha256": "ed5b93df2c3a59d473ddea853553a6d545d52bd4e9f8f72bf40b8a974aba4c1d"
    },
    {
      "path": "model.safetensors",
      "bytes": 115317516,
      "sha256": "8f1aa73170102c038d40155e2734b343bf07e0fe12594228a8590943b01dccf7"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 273,
      "sha256": "86a8837ae440456b0a9aef788b064921df29c20f0b67040954ce5c2fbd352c4f"
    }
  ],
  "totalBytes": 115320191
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = TableTransformerDetectionPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Render the synthetic page or optional BYOD

The default sample is **synthetic** and carries its own reference boxes: a deterministic 850×1100 page is rendered in code with Pillow's bundled font — a heading, two paragraphs of words, a wide 8×5 ruled table at `[70, 330, 780, 660]` and a small 5×3 ruled table at `[70, 760, 430, 990]`, every cell holding a short token, the way a PDF render looks. This is the same page the repository's smoke run used. The drawn boxes are the reference for the `box_iou` sanity check later; they are not a labelled dataset, so nothing here is a precision/recall measurement. The image digest is printed for the record. BYOD is optional and disabled by default; when enabled, upload one page image — no reference boxes exist for it, so the evaluation report will be `not-measurable`.

The detection threshold is a **caller-owned request parameter**, not a pipeline constant: a query survives when its softmax score for `table` or `table rotated` reaches it. The package default (`DETECTION_THRESHOLD = 0.9`) follows the Transformers documentation example, not a calibration; it is exposed here as a form parameter and passed explicitly on every call. Nothing is validated in this cell — the next section hands the image and the threshold to the pipeline's own validation stage, which is the only checker. Look for a dictionary naming the sample kind, the image size and digest, the threshold, and the drawn reference boxes.

In [ ]:
import hashlib
import io

import numpy as np
from PIL import Image, ImageDraw, ImageFont

USE_BYOD = False  # @param {type:"boolean"}
threshold = 0.9  # @param {type:"number"}


def synthetic_page(width=850, height=1100):
    """A letter-size page rendered with Pillow's bundled font: heading, two paragraphs, two ruled tables."""
    page = Image.new('RGB', (width, height), 'white')
    d = ImageDraw.Draw(page)
    body, head = ImageFont.load_default(size=15), ImageFont.load_default(size=22)
    words = 'quarterly revenue by region and product line for the fiscal year with notes on methodology'.split()
    d.text((70, 50), 'Annual Report: Regional Results', fill='black', font=head)
    y = 95
    for _para in range(2):
        for line in range(6):
            text = ' '.join(words[(line * 3 + k) % len(words)] for k in range(11 - (line % 3)))
            d.text((70, y), text, fill=(40, 40, 40), font=body)
            y += 20
        y += 16
    boxes = []
    for (x0, y0, x1, y1, rows, cols, first) in ((70, 330, 780, 660, 8, 5, 'Region'), (70, 760, 430, 990, 5, 3, 'Item')):
        d.rectangle([x0, y0, x1, y1], outline='black', width=2)
        rh, cw = (y1 - y0) / rows, (x1 - x0) / cols
        d.line([(x0, y0 + rh), (x1, y0 + rh)], fill='black', width=2)
        for r in range(2, rows):
            d.line([(x0, y0 + rh * r), (x1, y0 + rh * r)], fill=(120, 120, 120), width=1)
        for c in range(1, cols):
            d.line([(x0 + cw * c, y0), (x0 + cw * c, y1)], fill=(120, 120, 120), width=1)
        for r in range(rows):
            for c in range(cols):
                token = (first if c == 0 else f'Q{c}') if r == 0 else (f'North {r}' if c == 0 else f'{(r * 7 + c * 13) % 97 + 1},{(r * 31 + c) % 900 + 100:03d}')
                d.text((x0 + cw * c + 8, y0 + rh * r + rh / 2 - 8), token, fill='black', font=body)
        boxes.append([float(x0), float(y0), float(x1), float(y1)])
    d.text((70, 1010), 'Table 2 summarises the line items; see the appendix for the full breakdown.', fill=(40, 40, 40), font=body)
    return page, boxes


if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    image_name = next(iter(uploaded))
    image = Image.open(io.BytesIO(uploaded[image_name]))
    image.load()
    drawn_boxes = None
    sample_kind = 'BYOD'
else:
    # Deterministic synthetic page: no randomness, so no seed is needed and the digest is stable per Pillow build.
    image, drawn_boxes = synthetic_page()
    image_name = 'synthetic_page_850x1100.png'
    sample_kind = 'synthetic'

image_sha256 = hashlib.sha256(np.asarray(image.convert('RGB')).tobytes()).hexdigest()
print({'sample_kind': sample_kind, 'name': image_name, 'mode': image.mode, 'size': image.size, 'rgb_sha256': image_sha256, 'threshold': threshold, 'drawn_boxes': drawn_boxes})

## 5. Validate the request → input manifest

`validate_inputs` is the pipeline's public validation stage: it applies exactly the checks `detect` applies — image type and sides `MIN_IMAGE_SIDE`..`MAX_IMAGE_SIDE` px and a threshold in `[0, 1]` — and returns an **input manifest** naming the schema (including the two labels and the 15-query ceiling on detections), the input's observed mode and size, the threshold, and the verdict. The manifest is written to `outputs/table_transformer_detection_input_manifest.json`. To show what rejection looks like, the cell also validates a threshold outside `[0, 1]` and records the pipeline's own error message as a finding. Inside the pipeline the image is converted to RGB and resized to 800 px by the processor; boxes are mapped back to input pixels, and nothing else is dropped or altered.

In [ ]:
import json
import os

os.makedirs('outputs', exist_ok=True)
print({'ceilings': {'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_DETECTIONS': MAX_DETECTIONS, 'LABELS': list(LABELS), 'DETECTION_THRESHOLD': DETECTION_THRESHOLD}})
input_manifest = validate_inputs(image, threshold=threshold, names=[image_name])
# Demonstrate rejection on a request that breaks a ceiling; the finding is recorded, not swallowed.
try:
    validate_inputs(image, threshold=1.5)
except ValueError as exc:
    input_manifest['findings'].append({'input': 'out-of-range-threshold-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/table_transformer_detection_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest, indent=2))

## 6. Detect and read the scores correctly

`detect` returns a dict with `detections` — a list of `{box, label, score}` **ordered by descending score**, `box` in xyxy pixel coordinates of the input, `label` one of `table` / `table rotated` — plus the threshold used, `width`, `height` and the model identity. At most 15 boxes can ever be returned (the DETR decoder has 15 queries). Each `score` is the query's **softmax class probability under the model's own head, not a calibrated estimate for your pages**: it was never fitted to the frequency with which a box is a real table on your documents. The threshold you passed is the only decision rule; the pipeline ships 0.9 as a default, not as a calibration, and the caller owns it per deployment — lower it when a missed table costs more than a spurious box, raise it when a false table triggers downstream extraction. Inference is deterministic on a fixed device and dtype (no sampling, `torch.inference_mode`); CUDA kernel selection can move scores in the third or fourth decimal place. As recorded in the model card, the repository's CPU smoke on this same page at threshold 0.9 returned two `table` boxes at scores 0.999 and 0.997 with `box_iou` 0.80 and 0.66 against the drawn rectangles (the model's boxes hug the ruled area more tightly than the drawn outline); that is one observation, not a calibration point.

In [ ]:
result = pipe.detect(image, threshold=threshold)
print({'n_detections': len(result['detections']), 'threshold': result['threshold'], 'device': pipe.device})
for rank, det in enumerate(result['detections'], start=1):
    print(f"{rank:>2}. score {det['score']:.4f}  label {det['label']!r}  box {[round(v, 1) for v in det['box']]}")

## 7. Evaluate → evaluation report

`evaluation_report` is the pipeline's public evaluation stage and always produces a report. No detection metric is reported by default: mean average precision needs a labelled page set, and this repository ships none. The repository's only metric helper is `box_iou(a, b)` (intersection-over-union of two xyxy boxes), the building block a caller would use to compute mAP on their own labelled pages; when reference boxes are supplied the report carries one `box_iou` entry per reference — its value and which detection matched it best — with the verdict `sample-sanity`. On the synthetic path those references are tables **you rendered yourself**, so a high IoU proves only that the input contract, forward pass and coordinate mapping round-trip. On BYOD no reference exists, the verdict is `not-measurable`, and the report states what would make the task measurable: labelled table boxes on your own pages. The report is written to `outputs/table_transformer_detection_evaluation_report.json`.

In [ ]:
report = evaluation_report(result, drawn_boxes, sample_kind=sample_kind)
with open('outputs/table_transformer_detection_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps(report, indent=2))
if report['verdict'] == 'not-measurable':
    print('No reference boxes exist for this input, so box_iou is not computed; inspect the annotated PNG instead.')

## 8. Export outputs and provenance

Machine-readable JSON preserves the full result (score-ordered detections with boxes and labels, the threshold), the evaluation report, the input manifest, the sample identity, digest and drawn boxes, the notebook's source (repository, revision, embedded module digest, generator), the model identifier, the immutable model revision, the model licence, and the runtime identity (Python, `torch`, `transformers`, `timm`, device). The detections are also written as CSV with explicit `image`, `rank`, `label`, `score`, `x0`, `y0`, `x1`, `y1` columns so score ordering survives downstream use, and an annotated PNG draws every returned box for visual inspection (a supplement to, not a replacement for, the machine-readable files). No credentials are recorded.

In [ ]:
import csv

annotated = image.convert('RGB').copy()
draw = ImageDraw.Draw(annotated)
for det in result['detections']:
    draw.rectangle(det['box'], outline=(0, 160, 0), width=3)
    draw.text((det['box'][0] + 4, det['box'][1] + 4), f"{det['label']} {det['score']:.3f}", fill=(0, 160, 0))
annotated.save('outputs/table_transformer_detection_annotated.png')
payload = {
    'prediction': result,
    'evaluation_report': report,
    'input_manifest': input_manifest,
    'sample': {'kind': sample_kind, 'name': image_name, 'size': list(image.size), 'rgb_sha256': image_sha256, 'drawn_boxes': drawn_boxes},
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'timm': timm.__version__,
        'device': pipe.device,
    },
}
with open('outputs/table_transformer_detection_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
with open('outputs/table_transformer_detection_detections.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['image', 'rank', 'label', 'score', 'x0', 'y0', 'x1', 'y1'])
    for rank, det in enumerate(result['detections'], start=1):
        writer.writerow([image_name, rank, det['label'], f"{det['score']:.6f}", *[f"{v:.2f}" for v in det['box']]])
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The boxes locate regions the model classifies as a table on a page image; the label is one of two classes and the softmax score is not calibrated for your documents. The threshold is a request parameter you own; the default is the documentation example, not a tuned operating point. On the synthetic page the `box_iou` values in the evaluation report compare detections to tables you rendered yourself and the verdict is `sample-sanity`, which proves only that the input contract, forward pass and coordinate mapping work; they say nothing about scans, photographed pages, borderless tables, multi-column layouts, or non-Latin documents, and a BYOD result is a single-page observation with the verdict `not-measurable`. Everything is resized to 800 px on the shortest edge, so tables that are tiny at that scale may be missed. The pipeline provides no structure recognition, OCR, rotation correction, mAP evaluation, or training capability.

Successful execution proves that the recorded repository revision's pipeline module, carried in this notebook, can acquire and digest-verify the pinned model, validate the demonstrated request, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

**Next experiments:** lower `threshold` to 0.5 and count how many extra boxes appear on the same page (the smoke run found none, but a scan behaves differently); remove the ruling lines from one table in `synthetic_page` and see whether the score survives on text alignment alone; enable `USE_BYOD` with a scanned page, hand-label its table boxes and pass them to `evaluation_report` to see the verdict switch to `sample-sanity` — the first step towards a real precision/recall number; then feed a detected crop to the sibling structure-recognition pipeline.

## References

- Repository README: https://github.com/kurtvalcorza/table-transformer-detection-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/table-transformer-detection-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/table-transformer-detection-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/microsoft/table-transformer-detection
- Upstream code: https://github.com/microsoft/table-transformer
- PubTables-1M: Towards Comprehensive Table Extraction From Unstructured Documents (Smock, Pesala, Abraham, 2021): https://arxiv.org/abs/2110.00061